In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

from pyspark.sql import functions as F  
from pyspark.sql.types import StringType, StructType, StructField, TimestampType  
from pyspark.sql.utils import AnalysisException  

# Function: Securely retrieve S3 credentials from Databricks secret scope
def get_s3_credentials():
    """
    Purpose: Retrieve S3 access_key and secret_key from Databricks secret scope 'aws_keys'
    Arguments: None
    Returns: Tuple (access_key: str, secret_key: str)
    """
    try:
        access_key = dbutils.secrets.get(scope="aws_keys", key="access_key")
        secret_key = dbutils.secrets.get(scope="aws_keys", key="secret_key")
        return access_key, secret_key
    except Exception as e:
        raise RuntimeError(f"Missing S3 credentials in secret scope aws_keys: {str(e)}")

# Function: Fetch all active S3 landing paths and delimiters for patient_data from ingest_config_master
def get_patient_data_s3_configs():
    """
    Purpose: Retrieve all active S3 landing paths and delimiters for patient_data from ingest_config_master
    Arguments: None
    Returns: List of dicts: [{'s3_landing_path': str, 'delimiter': str}]
    """
    df = spark.table("purgo_playground.ingest_config_master") \
        .filter((F.col("source_object_name").contains("patient_data")) & (F.col("active_flag") == "Y"))
    configs = []
    rows = df.select("s3_landing_path", "delimiter").where(F.col("s3_landing_path").isNotNull()).collect()
    for row in rows:
        s3_path = row["s3_landing_path"]
        delimiter = row["delimiter"] if row["delimiter"] else ","
        configs.append({"s3_landing_path": s3_path, "delimiter": delimiter})
    return configs

# Function: Define schema for patient data CSV files
def get_patient_data_schema():
    """
    Purpose: Return the schema for patient data CSV files (all StringType)
    Arguments: None
    Returns: StructType schema
    """
    return StructType([
        StructField("patient_id", StringType(), True),
        StructField("patient_name", StringType(), True),
        StructField("age", StringType(), True),
        StructField("diagnosis", StringType(), True),
        StructField("treatment", StringType(), True)
    ])

# Function: Validate that DataFrame matches the target table schema (column names and types only)
def validate_schema(df):
    """
    Purpose: Ensure DataFrame columns and types match purgo_playground.patient_data_auto_loader schema
    Arguments:
        df (DataFrame): Input DataFrame
    Returns: None (raises AssertionError if schema mismatch)
    """
    expected_cols = ["patient_id", "patient_name", "age", "diagnosis", "treatment"]
    df_cols = df.columns
    if set(expected_cols) != set(df_cols):
        raise ValueError(f"Column mismatch: expected {expected_cols}, got {df_cols}")
    for col in expected_cols:
        if not isinstance(df.schema[col].dataType, StringType):
            raise TypeError(f"Column {col} must be StringType")

# Function: Add data_loaded_at column with current timestamp
def add_data_loaded_at(df):
    """
    Purpose: Add data_loaded_at column as current timestamp to DataFrame
    Arguments:
        df (DataFrame): Input DataFrame
    Returns: DataFrame with data_loaded_at (TimestampType)
    """
    return df.withColumn("data_loaded_at", F.current_timestamp().cast(TimestampType()))

# Function: Stream patient data from S3 using Auto Loader and append to Delta table
def stream_patient_data_to_delta(s3_path, delimiter, schema, checkpoint_location, schema_location):
    """
    Purpose: Stream patient data CSV files from S3 using Auto Loader and append to purgo_playground.patient_data_auto_loader
    Arguments:
        s3_path (str): S3 folder path
        delimiter (str): CSV delimiter
        schema (StructType): Schema for CSV
        checkpoint_location (str): Checkpoint location
        schema_location (str): Schema location
    Returns: StreamingQuery
    """
    df_stream = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("delimiter", delimiter) \
        .option("header", "true") \
        .option("cloudFiles.schemaLocation", schema_location) \
        .schema(schema) \
        .load(s3_path)
    # Validate schema (column names and types)
    validate_schema(df_stream)
    # Add data_loaded_at column
    df_stream = add_data_loaded_at(df_stream)
    # Select columns in correct order and types
    df_stream = df_stream.select(
        F.col("patient_id").cast(StringType()),
        F.col("patient_name").cast(StringType()),
        F.col("age").cast(StringType()),
        F.col("diagnosis").cast(StringType()),
        F.col("treatment").cast(StringType()),
        F.col("data_loaded_at").cast(TimestampType())
    )
    # Write stream to Delta table
    query = df_stream.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_location) \
        .toTable("purgo_playground.patient_data_auto_loader")
    return query

# Main execution block
# Securely retrieve S3 credentials
access_key, secret_key = get_s3_credentials()
# Configure Spark for S3 access
spark.conf.set("fs.s3a.access.key", access_key)
spark.conf.set("fs.s3a.secret.key", secret_key)
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")
# Retrieve all active patient_data S3 configs
configs = get_patient_data_s3_configs()
# If no configs found, print warning and exit gracefully
if not configs:
    print("Warning: No active patient_data config found in ingest_config_master. Exiting script gracefully.")
else:
    # Define schema, checkpoint, and schema locations
    schema = get_patient_data_schema()
    checkpoint_location = "/mnt/checkpoints/patient_al_cp/"
    schema_location = "/mnt/checkpoints/s3_autoloader/patient_al_schema"
    # For each config, start a streaming query
    queries = []
    for config in configs:
        s3_path = config["s3_landing_path"]
        delimiter = config["delimiter"]
        try:
            query = stream_patient_data_to_delta(
                s3_path=s3_path,
                delimiter=delimiter,
                schema=schema,
                checkpoint_location=checkpoint_location,
                schema_location=schema_location
            )
            queries.append(query)
        except Exception as e:
            # Log error and continue with other configs
            print(f"Error starting stream for {s3_path}: {str(e)}")
    # Optionally, wait for all queries to terminate (if running in a job context)
    # for query in queries:
    #     query.awaitTermination()
# End of script
